# HPD 重新触发 + 监视输入

背景：笔记本刚才**短暂检测到**过显示器，说明热插拔（HPD）是通的，
但 EDID 没读成功，Windows 就把它移除了。PYNQ 的驱动注释明确要求：
**EDID 就绪后再切换 HPD**，源端才会重新读 EDID。

这一格做三件事：
1. 加载 overlay；
2. 把 HPD 拉低 1 秒再拉高（制造一次干净的插拔事件）；
3. 连续监视 60 秒，看 `mode` 会不会变成 1280x720。

**运行时请同时盯着 Windows 的“显示设置”**：如果那里出现第二块屏，就是通了。


In [ ]:
import time
from pynq import Overlay

STOCK = "/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/base/base.bit"
ol = Overlay(STOCK)
video = ol.video
hpd = None
for name in ol.ip_dict:
    if "gpio_hdmiin" in name.lower():
        hpd = getattr(ol, name)
        print("HPD gpio:", name)
print("mode before:", video.hdmi_in.mode)


## 制造一次插拔事件，然后监视 60 秒

In [ ]:
if hpd is not None:
    hpd.mmio.write(0x00, 0x0)      # HPD low  = "cable unplugged"
    print("HPD -> 0 (unplugged)")
    time.sleep(1.5)
    hpd.mmio.write(0x00, 0x1)      # HPD high = "cable plugged in"
    print("HPD -> 1 (plugged), watching for a lock...")
else:
    print("no HPD gpio found; just watching")

locked = None
for i in range(60):
    m = video.hdmi_in.mode
    if m is not None:
        locked = m
        print(f"[{i:2d}] LOCKED {m.width}x{m.height} @{m.frame_rate} Hz")
        break
    if i % 5 == 0:
        print(f"[{i:2d}] mode: None")
    time.sleep(1.0)

if locked is None:
    print()
    print("60 秒内没有锁上。看 Windows 显示设置有没有出现第二块屏：")
    print("  有 -> EDID 通了但 TMDS 没锁，多半是分辨率/刷新率问题")
    print("  没有 -> EDID 都没读到，问题在 DDC：换线优先")
